In [ ]:
%pip install openai-agents pydantic mermaid-py nest-asyncio --upgrade

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# NOTE - This is only required for Jupyter notebook environments:
from nest_asyncio import apply
apply()

# Introduction to the OpenAI Agents SDK

The OpenAI Agents SDK (`openai-agents`) is a lightweight framework that eliminates the boilerplate of building agentic applications. In the previous notebook you built a manual agent loop from scratch: a `while` loop, a dispatch dictionary, `function_call_output` messages, and careful bookkeeping. The SDK handles all of that for you, so you can focus on defining agents, tools, and orchestration logic.

## Why Use an SDK?

In the Multi-Source Customer Support Agent notebook, you wrote ~20 lines of boilerplate to run a single agent:

```python
# MANUAL APPROACH (what you built in the previous notebook)
def run_agent(user_query, max_turns=10):
    messages = [{"role": "user", "content": user_query}]
    number_of_turns = 0
    
    while number_of_turns <= max_turns:
        response = client.responses.create(
            model=MODEL, instructions=instructions,
            input=messages, tools=tools
        )
        number_of_turns += 1
        tool_calls = [item for item in response.output if item.type == "function_call"]
        if not tool_calls:
            return response.output_text
        messages.extend(response.output)
        for tool_call in tool_calls:
            args = json.loads(tool_call.arguments)
            result = dispatch_tool_call(tool_call.name, args)
            messages.append({"type": "function_call_output", "call_id": tool_call.call_id, "output": result})
    return response.output_text
```

With the SDK, the same thing becomes:

```python
# SDK APPROACH (what you'll learn in this notebook)
from agents import Agent, Runner, function_tool

agent = Agent(name="Support", instructions=instructions, tools=[search_orders, search_products])
result = Runner.run_sync(agent, "What's the status of order #1003?")
print(result.final_output)
```

The SDK handles the while loop, tool dispatch, argument parsing, function_call_output messages, and turn counting. You just define your agents and tools.

## Setup

In [2]:
import getpass
import os

# The SDK reads OPENAI_API_KEY from the environment automatically.
# No need to instantiate an OpenAI() client.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print("OPENAI_API_KEY is set. Ready to go.")


OPENAI_API_KEY is set. Ready to go.


## Part 1: Your First Agent

An `Agent` is a large language model configured with a name, instructions, and optional tools. You run it with `Runner.run_sync()` (synchronous) or `await Runner.run()` (async).

In [7]:
from agents import Agent, Runner

agent = Agent(
    name="Haiku Writer",
    instructions="You are a creative poet. Always respond with a haiku (5-7-5 syllable pattern).",
    model="gpt-5-mini",
)

result = Runner.run_sync(agent, "Write about Python programming.")
print(result.final_output)

Indent guides my code
Libraries craft simple tools
Zen whispers, clean code


`Runner.run_sync()` handles the full agent loop internally. The `result` object contains:
- `result.final_output` — the agent's final text response
- `result.new_items` — all items (messages, tool calls) generated during the run
- `result.last_agent` — the agent that produced the final output (useful with handoffs)

## Part 2: Custom Tools with `@function_tool`

In the previous notebook, you defined tools using Pydantic `ToolArgs` models + a `function_tool()` helper + a dispatch dictionary. The SDK replaces all of that with a single `@function_tool` decorator.

The decorator:
1. Auto-generates the JSON schema from your function's type hints
2. Uses the docstring as the tool description
3. Handles argument parsing and dispatch automatically

In [8]:
from agents import function_tool


@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: The city name to check weather for.
    """
    # In production, this would call a weather API.
    weather_data = {
        "london": "Cloudy, 14°C",
        "tokyo": "Sunny, 28°C",
        "new york": "Partly cloudy, 22°C",
    }
    return weather_data.get(city.lower(), f"No weather data available for {city}")


# The decorator auto-generates the JSON schema:
print(f"Tool name: {get_weather.name}")
print(f"Tool description: {get_weather.description}")
print(f"Parameters schema: {get_weather.params_json_schema}")

Tool name: get_weather
Tool description: Get the current weather for a city.
Parameters schema: {'properties': {'city': {'description': 'The city name to check weather for.', 'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weather_args', 'type': 'object', 'additionalProperties': False}


In [11]:
weather_agent = Agent(
    name="Weather Assistant",
    instructions="You help users check the weather. Use the get_weather tool.",
    model="gpt-5-mini",
    tools=[get_weather],
)

result = Runner.run_sync(weather_agent, "What's the weather like in Tokyo?")
print(result.final_output)

Tokyo: Sunny, 28°C.


Compare this to the manual approach:

| Manual (previous notebook) | SDK (this notebook) |
|---|---|
| Define `ToolArgs` Pydantic model | Just use type hints on the function |
| Write `function_tool()` helper to generate schema | `@function_tool` decorator does it |
| Build `TOOL_FUNCTIONS` dispatch dict | SDK handles dispatch automatically |
| Write `while` loop with `function_call_output` messages | `Runner.run_sync()` handles the loop |
| Parse `json.loads(tool_call.arguments)` | SDK parses arguments for you |

## Part 3: Built-in Tools

The SDK provides ready-made wrappers for OpenAI's hosted tools. Instead of writing `{"type": "web_search_preview"}`, you use `WebSearchTool()`. Instead of `{"type": "code_interpreter"}`, you use `CodeInterpreterTool()`.

These tools run on OpenAI's servers, so there is nothing to implement on your side.

In [12]:
from agents import WebSearchTool

research_agent = Agent(
    name="Web Researcher",
    instructions="You research topics using web search. Provide concise, factual answers with sources.",
    model="gpt-5-mini",
    tools=[
        WebSearchTool(search_context_size="medium"),
    ],
)

result = Runner.run_sync(research_agent, "What is the latest version of the OpenAI Agents SDK?")
print(result.final_output)

Do you mean the Python or the JavaScript/TypeScript SDK? Short answer — as of today (March 5, 2026):

- Python SDK: v0.10.4 (released Mar 3, 2026). ([github.com](https://github.com/openai/openai-agents-python))  
- JavaScript/TypeScript SDK: v0.5.4 (released Mar 5, 2026). ([github.com](https://github.com/openai/openai-agents-js/releases))

Would you like the exact install commands or links to the release notes?


In [18]:
import pprint

for item in result.new_items:
    print(pprint.pformat(item))

ReasoningItem(agent=Agent(name='Web Researcher',
                          instructions='You research topics using web search. '
                                       'Provide concise, factual answers with '
                                       'sources.',
                          handoff_description=None,
                          handoffs=[],
                          model='gpt-5-mini',
                          model_settings=ModelSettings(temperature=None,
                                                       top_p=None,
                                                       frequency_penalty=None,
                                                       presence_penalty=None,
                                                       tool_choice=None,
                                                       parallel_tool_calls=None,
                                                       truncation=None,
                                                       max_tokens=None,
        

## Part 4: Structured Output with `output_type`

In previous notebooks, getting structured JSON output required:
1. Defining a Pydantic model
2. Calling `.model_json_schema()` to generate the schema
3. Passing `text={"format": {"type": "json_schema", "name": "...", "strict": True, "schema": ...}}`
4. Parsing with `Model.model_validate_json(response.output_text)`

With the SDK, you just pass `output_type=MyModel` and `result.final_output` is already a parsed Pydantic instance.

In [20]:
from pydantic import BaseModel, Field


class MovieRecommendation(BaseModel):
    """A structured movie recommendation."""
    title: str = Field(description="The movie title")
    year: int = Field(description="Release year")
    genre: str = Field(description="Primary genre")
    reason: str = Field(description="Why this movie is recommended")
    rating: float = Field(description="Rating out of 10")


recommender = Agent(
    name="Movie Recommender",
    instructions="Recommend a single movie based on the user's mood or preference.",
    model="gpt-5-mini",
    output_type=MovieRecommendation,
)

result = Runner.run_sync(recommender, "I want something mind-bending and thought-provoking.")

# result.final_output is already a MovieRecommendation instance!
movie = result.final_output
print(f"Title: {movie.title} ({movie.year})")
print(f"Genre: {movie.genre}")
print(f"Rating: {movie.rating}/10")
print(f"Reason: {movie.reason}")

Title: Coherence (2013)
Genre: Science fiction
Rating: 8.5/10
Reason: A low-key, tightly plotted sci-fi thriller about a dinner party that fractures into alternate realities. It’s mind-bending without relying on effects—focuses on identity, choice, and the unsettling consequences of parallel timelines, with an ambiguous ending that sparks discussion.


No manual JSON schema generation. No `model_validate_json()`. The SDK handles the entire structured output pipeline.

## Part 5: Agents as Tools

Sometimes you want one agent to delegate a sub-task to another agent **without transferring control** of the conversation. The `.as_tool()` method wraps an agent as a callable tool.

This differs from handoffs (Part 6) in two ways:
1. The sub-agent receives **generated input** (not the full conversation history)
2. The **original agent continues** the conversation after the tool call returns

In [25]:
spanish_translator = Agent(
    name="Spanish Translator",
    instructions="Translate the input text to Spanish. Return only the translation.",
    model="gpt-5-mini",
)

french_translator = Agent(
    name="French Translator",
    instructions="Translate the input text to French. Return only the translation.",
    model="gpt-5-mini",
)

from typing import Optional

class Translations(BaseModel):
    french_text: Optional[str] = Field(None)
    spanish_text: Optional[str] = Field(None)

orchestrator = Agent(
    name="Translation Orchestrator",
    instructions=(
        "You help users translate text. Use the available translation tools. "
        "Always provide translations in all available languages."
    ),
    model="gpt-5-mini",
    output_type=Translations,
    tools=[
        spanish_translator.as_tool(
            tool_name="translate_to_spanish",
            tool_description="Translate text to Spanish",
        ),
        french_translator.as_tool(
            tool_name="translate_to_french",
            tool_description="Translate text to French",
        ),
        WebSearchTool(search_context_size="medium"),
    ],
)

changed_query = '''I want you to do some web search for me on women's dresses. Then translate your research into both languages.'''
result = Runner.run_sync(orchestrator, changed_query)
print(result.final_output)

french_text="Résumé de recherche sur les robes pour femmes (concis) :\n\n1) Présentation des tendances 2026 : Les créateurs et rédacteurs de mode indiquent que les tendances des robes en 2026 mettent l'accent sur les tissus transparents et translucides (lingerie portée comme vêtement extérieur / look boudoir), les robes nuisette et de type nuisette, et le retour aux fleurs romantiques et aux pois ludiques — ainsi que des pastels doux tels que le lilas et des touches rouge tomate rapportées sur les podiums et dans les couvertures mode. \ue200cite\ue202turn0search3\ue202turn0search0\ue201\n\n2) Notes sur la longueur et la silhouette : Les longueurs plus longues sont à l'avant‑plan cette saison — les robes midi et maxi dominent, offrant des formes fluides et confortables qui allient aisance et élégance. \ue200cite\ue202turn0search9\ue202turn0search2\ue201\n\n3) Types de robes courants : Les silhouettes populaires incluent la robe trapèze (A‑line), la robe fourreau (sheath), la robe portef

In [26]:
model = result.final_output

In [28]:
model.spanish_text

'Resumen de investigación sobre vestidos de mujer (conciso):\n\n1) Panorama de tendencias 2026: Diseñadores y editores de moda dicen que las tendencias de vestidos para 2026 enfatizan tejidos transparentes y translúcidos (lencería como ropa exterior/estilo boudoir), vestidos tipo slip y similares, y un retorno a los estampados florales románticos y a los lunares juguetones, junto con pasteles suaves como el lila y acentos rojo tomate reportados en pasarelas y en la cobertura de estilo. \ue200cite\ue202turn0search3\ue202turn0search0\ue201\n\n2) Notas sobre largo y silueta: Los largos más extensos son prominentes esta temporada: los vestidos midi y maxi dominan, ofreciendo formas fluidas y cómodas que combinan comodidad con elegancia. \ue200cite\ue202turn0search9\ue202turn0search2\ue201\n\n3) Tipos comunes de vestidos: Las siluetas populares incluyen en A, vestido tubo (sheath), vestido cruzado (wrap), vestido recto (shift), vestido de verano (sundress), vestido camisa (shirtdress) y ves

The orchestrator called both translator agents as tools, collected their outputs, and composed a final response. The translator agents never saw the full conversation; they only received the text to translate.

## Part 6: Handoffs

Handoffs are the other way to build multi-agent systems. Unlike agents-as-tools, a handoff **transfers control** of the conversation to a different agent. The new agent receives the full conversation history and takes over completely.

Handoffs are exposed to the LLM as tools named `transfer_to_<agent_name>`. The model decides when to hand off based on its instructions.

In [29]:
billing_agent = Agent(
    name="Billing Agent",
    instructions="You handle billing questions. Explain pricing plans and payment options clearly.",
    model="gpt-5-mini",
)

tech_support_agent = Agent(
    name="Tech Support Agent",
    instructions="You handle technical issues. Help users troubleshoot problems step by step.",
    model="gpt-5-mini",
)

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You are the first point of contact. Determine what the user needs and "
        "hand off to the appropriate specialist agent. "
        "For billing questions, hand off to the Billing Agent. "
        "For technical issues, hand off to the Tech Support Agent."
    ),
    model="gpt-5-mini",
    handoffs=[billing_agent, tech_support_agent],
)

# Billing question -> should hand off to Billing Agent
result = Runner.run_sync(triage_agent, "How much does the Pro plan cost?")
print(f"Handled by: {result.last_agent.name}")
print(f"Answer: {result.final_output}")

Handled by: Billing Agent
Answer: Transferring you to the Billing Agent now for exact pricing and payment options.


In [30]:
# Technical question -> should hand off to Tech Support Agent
result = Runner.run_sync(triage_agent, "My API calls are returning 429 errors. Help!")
print(f"Handled by: {result.last_agent.name}")
print(f"Answer: {result.final_output}")

Handled by: Tech Support Agent
Answer: I'm handing this over to our Tech Support Agent who can look into your account and logs. To help them diagnose faster, please paste the following (or grant access if requested):

- Exact API endpoint(s) you're calling and approximate request rate (requests/sec).
- Full HTTP response headers and body for a 429 response (especially Retry-After, x-rate-limit headers, request-id if present).
- A short code snippet or curl command that reproduces the problem.
- Time(s) (including timezone) when you saw the errors.
- Whether you're using an SDK or raw HTTPS, and which version of the SDK.
- Any recent changes (new deployments, traffic spikes, new endpoints).
- Your account/project ID or the email associated with the account (do NOT paste secrets or API keys).

In the meantime, try these immediate mitigations:
- Honor the Retry-After header and implement exponential backoff + jitter.
- Reduce request rate or batch requests if possible.
- Cache responses w

### Agents as Tools vs Handoffs

| Feature | `agent.as_tool()` | `handoffs=[agent]` |
|---|---|---|
| Input to sub-agent | Generated input only | Full conversation history |
| Control flow | Original agent continues | Sub-agent takes over |
| Best for | Sub-tasks (translate, classify, etc.) | Routing (triage, escalation, etc.) |

## Part 7: Multi-turn Conversations

To carry conversation state between turns, use `result.to_input_list()` to convert the previous run's output into input for the next turn.

In [31]:
assistant = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Reply concisely.",
    model="gpt-5-mini",
)

# List concatenation: [2]  + [2] = [2, 2]

# Turn 1
result = Runner.run_sync(assistant, "What city is the Golden Gate Bridge in?")
print(f"Turn 1: {result.final_output}")

# Turn 2 — pass previous context + new question
new_input = result.to_input_list() + [{"role": "user", "content": "What state is it in?"}]
result = Runner.run_sync(assistant, new_input)
print(f"Turn 2: {result.final_output}")

# Turn 3 — continue the conversation
new_input = result.to_input_list() + [{"role": "user", "content": "What is the population of that state?"}]
result = Runner.run_sync(assistant, new_input)
print(f"Turn 3: {result.final_output}")

Turn 1: The Golden Gate Bridge is in San Francisco, California — it spans the Golden Gate strait, connecting the City of San Francisco to Marin County (near Sausalito).
Turn 2: The Golden Gate Bridge is in California, USA.
Turn 3: As of the 2020 U.S. Census, California's population was 39,538,223. More recent estimates (2021–2023) put it at about 39.2 million people (U.S. Census Bureau annual estimates). Would you like the exact latest estimate for a specific year?


## Part 8: Tracing

The `trace()` context manager groups multiple agent runs into a single trace that you can view in the OpenAI dashboard. This is useful for debugging and understanding how your multi-agent system behaves.

In [32]:
from agents import trace

with trace("Translation Workflow"):
    # All Runner.run_sync calls inside this block are grouped into one trace
    result = Runner.run_sync(
        orchestrator,
        "Translate 'The weather is beautiful today' to all available languages.",
    )
    print(result.final_output)

print("\nTrace has been sent to the OpenAI dashboard.")
print("View it at: https://platform.openai.com/traces")

french_text="Il fait beau aujourd'hui." spanish_text='Hace buen tiempo hoy.'

Trace has been sent to the OpenAI dashboard.
View it at: https://platform.openai.com/traces


## Exercise: Build a Recipe Recommendation Agent

Build an agent that recommends recipes based on available ingredients. Your agent should have:

1. A `@function_tool` called `search_recipes` that searches a hardcoded recipe database by ingredient
2. An `output_type` of `RecipeRecommendation` (Pydantic model) with fields: `name`, `ingredients`, `cooking_time_minutes`, `difficulty`, `instructions`

The recipe database is provided for you below.

In [ ]:
import json

RECIPE_DATABASE = [
    {
        "name": "Pasta Carbonara",
        "ingredients": ["pasta", "eggs", "bacon", "parmesan", "black pepper"],
        "cooking_time": 25,
        "difficulty": "medium",
    },
    {
        "name": "Chicken Stir Fry",
        "ingredients": ["chicken", "soy sauce", "vegetables", "rice", "garlic"],
        "cooking_time": 20,
        "difficulty": "easy",
    },
    {
        "name": "Caesar Salad",
        "ingredients": ["lettuce", "croutons", "parmesan", "caesar dressing", "chicken"],
        "cooking_time": 10,
        "difficulty": "easy",
    },
    {
        "name": "Beef Tacos",
        "ingredients": ["ground beef", "taco shells", "lettuce", "cheese", "salsa"],
        "cooking_time": 15,
        "difficulty": "easy",
    },
    {
        "name": "Mushroom Risotto",
        "ingredients": ["rice", "mushrooms", "onion", "white wine", "parmesan"],
        "cooking_time": 40,
        "difficulty": "hard",
    },
]

print(f"Recipe database has {len(RECIPE_DATABASE)} recipes.")

In [ ]:
from agents import Agent, Runner, function_tool
from pydantic import BaseModel, Field


# Step 1: Define the output type
class RecipeRecommendation(BaseModel):
    """A structured recipe recommendation."""
    # YOUR CODE HERE: Add fields for name, ingredients (list[str]),
    # cooking_time_minutes (int), difficulty (str), instructions (str)
    pass


# Step 2: Define the search_recipes tool
# YOUR CODE HERE: Use @function_tool to create a search_recipes function
# It should accept an 'ingredient' parameter (str) and return matching recipes as JSON
# Hint: Search the RECIPE_DATABASE for recipes containing that ingredient


# Step 3: Create the agent
# YOUR CODE HERE: Create a recipe_agent with:
#   - name="Recipe Chef"
#   - instructions that tell it to search for recipes, then recommend the best match
#   - tools=[search_recipes]
#   - output_type=RecipeRecommendation


# Step 4: Test it
# result = Runner.run_sync(recipe_agent, "I have chicken and rice. What can I make?")
# print(f"Recipe: {result.final_output.name}")
# print(f"Time: {result.final_output.cooking_time_minutes} minutes")
# print(f"Difficulty: {result.final_output.difficulty}")
# print(f"Instructions: {result.final_output.instructions}")

## Summary

In this notebook you learned the core primitives of the OpenAI Agents SDK:

| SDK Primitive | What It Replaces | Example |
|---|---|---|
| `Agent(name, instructions, tools)` | Manual `client.responses.create()` config | Define an agent with behavior and capabilities |
| `Runner.run_sync(agent, input)` | The `while` loop + tool dispatch + `function_call_output` messages | Run an agent and get the result |
| `@function_tool` | Pydantic `ToolArgs` + `function_tool()` helper + dispatch dict | Define a tool from a plain Python function |
| `WebSearchTool()` | `{"type": "web_search_preview"}` | Built-in web search |
| `CodeInterpreterTool()` | `{"type": "code_interpreter"}` | Built-in code execution |
| `output_type=MyModel` | `text={"format": {"type": "json_schema", ...}}` + `model_validate_json()` | Structured output as a parsed Pydantic instance |
| `agent.as_tool()` | Manual sub-agent invocation | Use an agent as a tool without transferring control |
| `handoffs=[agent]` | Manual routing logic | Transfer conversation control to a specialist agent |
| `result.to_input_list()` | Manual message list management | Carry conversation state between turns |
| `trace("name")` | No equivalent | Group runs for observability in the OpenAI dashboard |